# gql

> GraphQL client: distill a schema once, then build schema-checked queries by attribute chaining, batched into single requests

In [ ]:
#| default_exp gql

In [ ]:
#| export
from fastcore.utils import *
from fastspec.transport import AsyncTransport

import json, asyncio

In [ ]:
import os, tempfile
from fastcore.test import *
from pyskills import xdir

In [ ]:
#| hide
from cachy.core import enable_cachy, doms

In [ ]:
#| hide
enable_cachy(doms=doms+('api.github.com', 'graphqlzero.almansi.me'), hdrs=('content-type',))

A GraphQL API is one endpoint that accepts a *shape*: describe the nesting you want and the server returns exactly that, in one round trip. The price is that you must know the schema to write the shape. This module closes that gap for any GraphQL endpoint: `GqlSpec` distills the standard introspection answer into compact tables a client package can ship, and `GqlClient` uses them to expose the schema for discovery (`xdir`, attribute completion, rich reprs), build queries by attribute chaining with plain-kwargs arguments, and execute many independent queries as batched requests. Raw GraphQL text works at every level, and only query fields are exposed as attributes -- mutations require deliberately writing raw text. Everything except execution is offline: distilling, discovery, and query building need no network.

## The distilled schema

Our example throughout is [GraphQLZero](https://graphqlzero.almansi.me/) -- a public GraphQL service over the classic JSONPlaceholder sample data (users, posts, albums), with no auth and a schema small enough to read whole. GitHub arrives at the end, for scale.

Every GraphQL server answers the standard introspection query. The raw answer is verbose, so `INTROSPECT` asks for just what a client needs -- fields with their args, types, defaults, and first-line descriptions:


In [ ]:
#| export
_TREF = 'kind name ofType { kind name ofType { kind name ofType { kind name ofType { kind name } } } }'
INTROSPECT = '''{
  __schema {
    queryType { name } mutationType { name }
    types {
      kind name description
      fields(includeDeprecated: false) {
        name description
        args { name description defaultValue type { %s } }
        type { %s }
      }
      inputFields { name description defaultValue type { %s } }
      enumValues { name description }
      possibleTypes { name }
    }
  }
}''' % (_TREF, _TREF, _TREF)

def _tstr(t):
    "Render an introspection type ref as GraphQL notation, e.g. '[Ref!]!'"
    if t['kind'] == 'NON_NULL': return _tstr(t['ofType']) + '!'
    if t['kind'] == 'LIST': return '[' + _tstr(t['ofType']) + ']'
    return t['name']

def _desc(s): return s.strip().split('\n', 1)[0].strip() if s else ''

In [ ]:
test_eq(_desc('First line.\nSecond line.'), 'First line.')
ref = dict(kind='OBJECT', name='Ref')
_tstr(dict(kind='NON_NULL', ofType=dict(kind='LIST', ofType=dict(kind='NON_NULL', ofType=ref))))

'[Ref!]!'

In [ ]:
#| export
def _fieldrec(f):
    "Compact record for one object field: rendered type, first-line doc, and per-arg [type, doc, default]"
    return dict(type=_tstr(f['type']), desc=_desc(f['description']),
        args={a['name']: [_tstr(a['type']), _desc(a['description']), a['defaultValue']] for a in f['args']})

def distill(raw):
    "Distill standard introspection JSON into compact lookup tables"
    sch = raw['data']['__schema']
    types = {}
    for t in sch['types']:
        nm = t['name']
        if nm.startswith('__'): continue
        d = dict(kind=t['kind'])
        if _desc(t['description']): d['desc'] = _desc(t['description'])
        if t['kind'] in ('OBJECT', 'INTERFACE'): d['fields'] = {f['name']: _fieldrec(f) for f in t['fields'] or []}
        elif t['kind'] == 'INPUT_OBJECT':
            d['fields'] = {f['name']: dict(type=_tstr(f['type']), desc=_desc(f['description']), default=f['defaultValue'])
                for f in t['inputFields'] or []}
        elif t['kind'] == 'ENUM': d['values'] = {v['name']: _desc(v['description']) for v in t['enumValues'] or []}
        elif t['kind'] == 'UNION': d['of'] = [p['name'] for p in t['possibleTypes'] or []]
        types[nm] = d
    return dict(query=sch['queryType']['name'], mutation=sch['mutationType']['name'], types=types)

Distilling the real thing -- the whole GraphQLZero schema, live, then one type's table. Wrapper types render in GraphQL notation (`String!`), each field carries its args as `[type, doc, default]`, and introspection meta-types are dropped:

In [ ]:
Z_URL = 'https://graphqlzero.almansi.me/api'
raw = await AsyncTransport().request('POST', Z_URL, json_data=dict(query=INTROSPECT))
tbls = distill(raw)
test_eq(tbls['query'], 'Query')
assert not any(t.startswith('__') for t in tbls['types'])
tbls['types']['Address']


{'kind': 'OBJECT',
 'fields': {'street': {'type': 'String', 'desc': '', 'args': {}},
  'suite': {'type': 'String', 'desc': '', 'args': {}},
  'city': {'type': 'String', 'desc': '', 'args': {}},
  'zipcode': {'type': 'String', 'desc': '', 'args': {}},
  'geo': {'type': 'Geo', 'desc': '', 'args': {}}}}

`GqlSpec` wraps the tables, mirroring `SpecParser`: build one with `from_introspection`, round-trip with `to_dict`/`from_dict`:

In [ ]:
#| export
class GqlSpec:
    "Distilled GraphQL schema: field/arg/type lookup tables, shippable via `save`"
    def __init__(self, query, mutation, types): self.query, self.mutation, self.types = query, mutation, types

    @classmethod
    def from_introspection(cls, raw):
        "Build from a standard introspection answer (see `INTROSPECT`)"
        return cls(**distill(raw))

    @classmethod
    def from_dict(cls, d): return cls(**d)
    def to_dict(self): return dict(query=self.query, mutation=self.mutation, types=self.types)

    def save(self, nm):
        "Write an importable module exposing these tables as `gqlspec`, one line per type"
        lines = [f'gqlspec = {{"query": {self.query!r}, "mutation": {self.mutation!r}, "types": {{']
        lines += [f'{k!r}: {v!r},' for k, v in self.types.items()]
        Path(nm).write_text('\n'.join(lines + ['}}']))

    @property
    def t(self): return _TypeIdx(self.types)

    def __repr__(self): return f'GqlSpec(query={self.query!r}, types={len(self.types)})'

In [ ]:
spec = GqlSpec.from_introspection(raw)
test_eq(GqlSpec.from_dict(spec.to_dict()).to_dict(), spec.to_dict())
spec

GqlSpec(query='Query', types=48)

`save` writes the tables as an importable module, one line per type so regeneration diffs stay readable. This is how a client package ships its schema and skips the runtime introspection entirely -- ghapi ships GitHub's this way as `ghapi.gql_spec`:

In [ ]:
fn = Path(tempfile.mkdtemp())/'zero_spec.py'
spec.save(fn)
ns = {}
exec(fn.read_text(), ns)
test_eq(GqlSpec.from_dict(ns['gqlspec']).to_dict(), spec.to_dict())
print(fn.read_text()[:120] + '...')

gqlspec = {"query": 'Query', "mutation": 'Mutation', "types": {
'Query': {'kind': 'OBJECT', 'fields': {'_': {'type': 'In...


Not everything sits on a path you are building: enum values, input-object shapes, and union membership are looked up rather than navigated to. `spec.t` indexes every type by name, each with a readable repr:

In [ ]:
#| export
class _TypeInfo:
    def __init__(self, types, nm): self._types, self._nm = types, nm
    def __dir__(self): return list(self._types[self._nm].get('fields', {}))
    def __repr__(self):
        t = self._types[self._nm]
        res = [f'{self._nm} ({t["kind"]})'] + ([t['desc']] if t.get('desc') else [])
        if t['kind'] == 'ENUM': res += [f'  {k}: {v}' if v else f'  {k}' for k, v in t['values'].items()]
        elif t['kind'] == 'UNION': res.append('one of: ' + ', '.join(t['of']))
        else: res += [f'  {k}: {f["type"]}' + (f'  # {f["desc"]}' if f['desc'] else '') for k, f in t.get('fields', {}).items()]
        return '\n'.join(res)

class _TypeIdx:
    def __init__(self, types): self._types = types
    def __dir__(self): return list(self._types)
    def __getattr__(self, k):
        if k.startswith('_') or k not in self._types: raise AttributeError(k)
        return _TypeInfo(self._types, k)

In [ ]:
spec.t.SortOrderEnum


SortOrderEnum (ENUM)
  ASC
  DESC

## The client

`GqlClient` executes queries, riding the same `AsyncTransport` as `OpenAPIClient` -- one place for timeouts, header merging, and HTTP errors enriched with the response body. Calling it with raw GraphQL text (plus optional variables) is the whole API in one line; the fragment layer below builds that text for you. GraphQL reports failures as an `errors` list that can arrive *alongside* partial data on a successful HTTP exchange, so failures raise `GqlError`, which keeps the structured list (`.errors`) and any partial payload (`.data`).

In [ ]:
#| export
class GqlError(Exception):
    "A GraphQL `errors` response; `errors` is the structured list, `data` any partial payload"
    def __init__(self, errors, data=None):
        self.errors, self.data = errors, data
        super().__init__('; '.join(e.get('message', str(e)) for e in errors))

class GqlClient:
    "GraphQL client over a `GqlSpec`: query fields as attributes, `batch` for one-request fan-out"
    def __init__(self, spec, url, *, headers=None, timeout=60.0):
        self.spec, self.url = spec, url
        self.transport = AsyncTransport(timeout=timeout, base_headers=headers)

    async def _post(self, query, **vars):
        return await self.transport.request('POST', self.url, json_data=dict(query=query, variables=vars or None))

    async def __call__(self, query, **vars):
        res = await self._post(query, **vars)
        if res.get('errors'): raise GqlError(res['errors'], res.get('data'))
        return dict2obj(res['data'])

    def __getattr__(self, k):
        if k.startswith('_') or k in ('spec', 'url', 'transport'): raise AttributeError(k)
        if k not in self.spec.types[self.spec.query]['fields']: raise AttributeError(f'no query field {k!r}')
        return GqlFrag(self, ((k, {}, None),))

    def __dir__(self): return list(self.spec.types[self.spec.query]['fields'])

    @property
    def t(self): return self.spec.t

In [ ]:
zq = GqlClient(spec, Z_URL)
res = await zq('query($id: ID!) { user(id: $id) { name email } }', id=1)
test_eq(res.user.name, 'Leanne Graham')
res


```python
{'user': {'email': 'Sincere@april.biz', 'name': 'Leanne Graham'}}
```

## Building queries

Queries are built as *fragments*: attribute access on the client starts a path at a query field, each further attribute extends it a field at a time, and calling a fragment binds arguments as plain kwargs. The seam rule: **args are kwargs; selection is attribute chaining when linear, raw GraphQL text when not.** Argument values render as GraphQL literals -- strings quoted, enums (recognized from the schema, including inside input objects) bare:

In [ ]:
#| export
def _basetype(tstr): return re.sub(r'[\[\]!]', '', tstr)

def _fmt_val(v, tstr, types):
    "Python value -> GraphQL literal; enum strings and input-object fields typed per the schema"
    base = types.get(_basetype(tstr), {})
    if isinstance(v, str) and base.get('kind') == 'ENUM': return v
    if isinstance(v, bool): return 'true' if v else 'false'
    if v is None: return 'null'
    if isinstance(v, (int, float)): return str(v)
    if isinstance(v, str): return json.dumps(v)
    if isinstance(v, (list, tuple)): return '[' + ', '.join(_fmt_val(o, tstr, types) for o in v) + ']'
    if isinstance(v, dict):
        flds = base.get('fields', {})
        return '{' + ', '.join(f'{k}: {_fmt_val(o, flds.get(k, {}).get("type", ""), types)}' for k, o in v.items()) + '}'
    raise TypeError(f'Cannot render {type(v)} as GraphQL literal')

def _fmt_args(args, argspec, types):
    if not args: return ''
    return '(' + ', '.join(f'{k}: {_fmt_val(v, argspec.get(k, [""])[0], types)}' for k, v in args.items()) + ')'

In [ ]:
test_eq(_fmt_val(True, 'Boolean', {}), 'true')
test_eq(_fmt_val(['a', 'b'], '[String!]', {}), '["a", "b"]')
test_eq(_fmt_val('ASC', 'SortOrderEnum', spec.types), 'ASC')
_fmt_val(dict(sort=[dict(field='title', order='ASC')]), 'PageQueryOptions', spec.types)


'{sort: [{field: "title", order: ASC}]}'

`GqlFrag` holds a path of `(field, args, raw-selection)` steps and walks the schema to know where it stands: attribute access is checked against the current type's fields, so a typo fails at chain time naming the type, and `__dir__` makes `xdir` (and tab completion) navigate the schema:

In [ ]:
#| export
class GqlFrag:
    "A lazily-built GraphQL query path: attribute access extends it, calling binds args, awaiting executes"
    def __init__(self, client, steps): self._c, self._steps = client, steps

    def _fieldinfo(self, i=None):
        "The (result type name, field spec) at step `i` (default last), walking from the query root"
        types = self._c.spec.types
        cur, fs = self._c.spec.query, None
        for nm, _, _ in (self._steps if i is None else self._steps[:i+1]):
            fs = types[cur]['fields'][nm]
            cur = _basetype(fs['type'])
        return cur, fs

    def _curtype(self): return self._fieldinfo()[0]
    def _isleaf(self):
        t = self._c.spec.types.get(self._curtype())
        return t is None or t['kind'] in ('SCALAR', 'ENUM')

    def __getattr__(self, k):
        if k.startswith('_'): raise AttributeError(k)
        if self._steps[-1][2] is not None: raise AttributeError('cannot chain past a raw selection')
        flds = self._c.spec.types[self._curtype()].get('fields', {})
        if k not in flds: raise AttributeError(f'{self._curtype()} has no field {k!r}; xdir this fragment to list fields')
        return GqlFrag(self._c, self._steps + ((k, {}, None),))

    def __call__(self, raw=None, **kw):
        nm, args, _ = self._steps[-1]
        return GqlFrag(self._c, self._steps[:-1] + ((nm, {**args, **kw}, raw),))

    def __dir__(self): return list(self._c.spec.types[self._curtype()].get('fields', {}))

In [ ]:
assert {'user', 'users', 'post', 'album'} <= set(dir(zq))
with expect_fail(AttributeError, contains='has no field'): zq.user.no_such_field
xdir(zq.user.address)


['city', 'geo', 'street', 'suite', 'zipcode']

Rendering walks the steps from the leaf outward, wrapping each in its parent's braces, with args formatted per the schema:

In [ ]:
#| export
@patch
def _qbody(self:GqlFrag):
    types, out = self._c.spec.types, ''
    for i in range(len(self._steps) - 1, -1, -1):
        nm, args, raw = self._steps[i]
        s = nm + _fmt_args(args, self._fieldinfo(i)[1]['args'], types)
        if raw: s += ' { ' + raw + ' }'
        elif out: s += ' { ' + out + ' }'
        out = s
    return out

@patch
def _q(self:GqlFrag): return '{ ' + self._qbody() + ' }'

The repr is the interface. An *unfinished* fragment (one whose current type still has fields to choose) displays as help -- signature, docs, and what you can chain next; a *complete* one displays as the query it will send, teaching the raw language as you go:

In [ ]:
#| export
@patch
def __repr__(self:GqlFrag):
    if self._isleaf() or self._steps[-1][2] is not None: return self._q()
    cur, fs = self._fieldinfo()
    nm = self._steps[-1][0]
    args = ', '.join(f'{k}: {v[0]}' + (f' = {v[2]}' if v[2] is not None else '') for k, v in fs['args'].items())
    res = [f'{nm}({args}) -> {fs["type"]}' if args else f'{nm} -> {fs["type"]}']
    if fs['desc']: res.append(fs['desc'])
    if argdocs := [f'  {k}: {v[1]}' for k, v in fs['args'].items() if v[1]]: res += ['args:'] + argdocs
    if flds := self._c.spec.types[cur].get('fields', {}):
        res.append(f'fields of {cur}: ' + ' '.join(list(flds)[:20]) + (' ...' if len(flds) > 20 else ''))
    return '\n'.join(res)

In [ ]:
zq.user


user(id: ID!) -> User
fields of User: id name username email address phone website company posts albums todos

In [ ]:
zq.user(id=1).address.geo.lat


{ user(id: 1) { address { geo { lat } } } }

Awaiting a fragment executes it and unwraps the answer along the path, so a chain to a scalar returns the bare value. A fragment that still needs a selection refuses before any network:

In [ ]:
#| export
def _unwrap(data, steps):
    for nm, _, raw in steps:
        if data is None: return None
        data = data[nm]
        if raw: break
    return data

@patch
def __await__(self:GqlFrag):
    async def _go():
        if not self._isleaf() and self._steps[-1][2] is None:
            raise TypeError(f'{self._curtype()} needs a selection: chain to a leaf field, or call with raw GraphQL text')
        return _unwrap(await self._c(self._q()), self._steps)
    return _go().__await__()

In [ ]:
res = await zq.user(id=1).address.geo.lat
test_eq(res, -37.3159)
with expect_fail(TypeError, contains='needs a selection'): await zq.user(id=1)
res


-37.3159

When a selection branches -- several fields at once, maybe nested -- chaining stops being the right notation, and raw GraphQL text takes over: call any node with a selection string. Combined with a nested input-object arg (note `ASC` bare beside quoted `"title"` in the generated query), one fragment can carry the whole request:


In [ ]:
f = zq.posts(options=dict(sort=[dict(field='title', order='ASC')], paginate=dict(limit=3)))('data { title }')
print(f)
await f


{ posts(options: {sort: [{field: "title", order: ASC}], paginate: {limit: 3}}) { data { title } } }


```python
{ 'data': [{'title': 'a quo magni similique perferendis'}, {'title': 'ad iusto omnis odit dolor voluptatibus'}, {'title': 'adipisci placeat illum aut reiciendis qui'}]}
```

Because a query is a shape, "run these N fragments" is just one bigger shape: `batch` aliases each fragment into a single request and returns results in input order, with `None` for any alias that errored rather than killing the other answers. Pass fragments individually, or as one iterable; `chunk=` (default `batch_chunk` on the client) splits large batches into parallel requests, for servers that resolve aliases serially:


In [ ]:
#| export
GqlClient.batch_chunk = None

@patch
async def _batch1(self:GqlClient, frags):
    res = await self._post('{ ' + ' '.join(f'r{i}: {f._qbody()}' for i, f in enumerate(frags)) + ' }')
    if res.get('data') is None or any(not e.get('path') for e in res.get('errors', [])):
        raise GqlError(res.get('errors', [dict(message='no data')]), res.get('data'))
    data = dict2obj(res['data'])
    return [None if data[f'r{i}'] is None else _unwrap({f._steps[0][0]: data[f'r{i}']}, f._steps) for i, f in enumerate(frags)]

@patch
async def batch(self:GqlClient, *frags, chunk=None):
    "Execute fragments (or one iterable of them) as aliased requests, `chunk` per request in parallel; results in input order, `None` where an alias errored"
    if len(frags) == 1 and not isinstance(frags[0], GqlFrag): frags = tuple(frags[0])
    chunk = ifnone(chunk, self.batch_chunk) or max(len(frags), 1)
    res = await asyncio.gather(*[self._batch1(frags[i:i+chunk]) for i in range(0, len(frags), chunk)])
    return [h for r in res for h in r]

In [ ]:
res = await zq.batch(zq.user(id=i).name for i in (1, 2, 3))
test_eq(res, ['Leanne Graham', 'Ervin Howell', 'Clementine Bauch'])
test_eq(await zq.batch((zq.user(id=i).name for i in (1, 2, 3)), chunk=2), res)
res


['Leanne Graham', 'Ervin Howell', 'Clementine Bauch']

Long lists page rather than batch: Relay-convention APIs (GitHub among them) expose them as *connection* fields walked with `first`/`after` cursors, each response minting the next cursor. `paged` follows the cursors to the end, yielding nodes as they arrive -- serial by protocol design, where `batch` fans out in parallel:

In [ ]:
#| export
@patch
async def paged(self:GqlClient, frag, select, per_page=100):
    "Yield each node of Relay-style connection fragment `frag`, selecting `select` per node, following cursors to the end"
    cursor = None
    while True:
        res = await frag(first=per_page, after=cursor)(f'pageInfo {{ endCursor hasNextPage }} nodes {{ {select} }}')
        for o in res['nodes']: yield o
        if not res['pageInfo']['hasNextPage']: return
        cursor = res['pageInfo']['endCursor']

## GitHub at scale

Now GitHub: the same machinery against a schema three hundred times bigger, authenticated with a `GITHUB_TOKEN`. The size drop from distilling is why clients ship the distilled form rather than introspecting at runtime -- ghapi ships exactly this as `ghapi.gql_spec`, regenerated at build time:


In [ ]:
GH_GQL = 'https://api.github.com/graphql'
ghhdrs = {'Authorization': f'bearer {os.environ["GITHUB_TOKEN"]}'}
raw = await AsyncTransport(base_headers=ghhdrs).request('POST', GH_GQL, json_data=dict(query=INTROSPECT))
ghspec = GqlSpec.from_introspection(raw)
print(f'raw introspection: {len(json.dumps(raw))/1e6:.1f}MB -> distilled: {len(json.dumps(ghspec.to_dict()))/1e6:.1f}MB')
ghspec

raw introspection: 2.4MB -> distilled: 1.3MB


GqlSpec(query='Query', types=1813)

The same `batch` at fleet scale: the head commit of many repos in one request -- checking which of a hundred repos moved costs one round trip. GitHub reports a missing repository as a path-scoped error alongside null data for its alias, so `batch` returns `None` there and real results for the rest (a *global* error -- bad syntax, auth -- still raises):


In [ ]:
ggql = GqlClient(ghspec, GH_GQL, headers=ghhdrs)
heads = await ggql.batch(*[ggql.repository(owner='AnswerDotAI', name=n).defaultBranchRef.target.oid
    for n in ('fastcore', 'no-such-repo-xyz', 'ghapi')])
test_eq(heads[1], None)
test_eq(len(heads[0]), 40)
heads

['25c4f3228ccac3c5a63da71b5eaa4be3c428f602',
 None,
 '4ca8469d7c2ccc42cb71e30a576c719f306b5cf7']

GitHub resolves a query's aliases *serially* (roughly 50ms per repository lookup), so one giant batch hits a wall: against a 103-repo workspace, a single 103-alias query measured 5.5s while four parallel 25-alias requests took 1.7s. Pass `chunk=` (or set `batch_chunk` on the client, as ghapi's `GhGql` does with its GitHub-tuned 25) and `batch` issues parallel chunked requests transparently:

In [ ]:
heads2 = await ggql.batch((ggql.repository(owner='AnswerDotAI', name=n).defaultBranchRef.target.oid
    for n in ('fastcore', 'no-such-repo-xyz', 'ghapi')), chunk=2)
test_eq(heads2[1], None)
heads2


['25c4f3228ccac3c5a63da71b5eaa4be3c428f602',
 None,
 'ad07893daac86de6693bc9bb57ae7216c1b347d0']

Depth is a different limit than breadth: every GitHub connection field caps `first:` at 100, and item #101 is reachable only through the previous response's cursor, so walking a long list is inherently serial. `paged` automates that walk for any [Relay-convention](https://relay.dev/graphql/connections.htm) connection (`nodes` plus `pageInfo`, which is how GitHub and most large GraphQL APIs paginate). Here it crosses the 100-repo cap in two requests:

In [ ]:
names = [o.name async for o in ggql.paged(ggql.organization(login='AnswerDotAI').repositories, 'name')]
assert len(names) > 100
len(names)

547

A solo await of the same missing repo raises instead, and `GqlError` keeps the structured report -- `.errors` with paths, `.data` with whatever partial payload arrived:

In [ ]:
try: await ggql.repository(owner='AnswerDotAI', name='no-such-repo-xyz').defaultBranchRef.target.oid
except GqlError as e: err = e
test_eq(err.errors[0]['type'], 'NOT_FOUND')
err.errors[0]['path'], err.data

(['repository'], {'repository': None})

## Export -

In [ ]:
#| hide
#| eval: false
import nbdev; nbdev.nbdev_export()